# Feature Matrix Tour

In this notebook I inspect the frozen transition-model feature matrices. Each row corresponds to one candidate plan, and each column is a transition-consistency statistic computed from a frozen source model.

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent

DATASET = ROOT / "data" / "correctness_dataset"
CANDIDATES = DATASET / "candidates"
FEATURES = DATASET / "features"
HEADS = ROOT / "models" / "correctness_heads"
RESULTS = ROOT / "results" / "analysis"

print(f"Repository root: {ROOT}")

Repository root: /home/bernardod/desktop/research/planfm-validity


In [11]:
family = "dd_xgb_wl_delta"
seed = 13
split = "train"
path = FEATURES / family / f"seed_{seed}" / f"{split}.npz"
feature_file = np.load(path, allow_pickle=True)

X = feature_file["X"]
correctness_scores = feature_file["correctness_scores"]
feature_names = [str(name) for name in feature_file["feature_names"]]

print(f"Loaded: {path.relative_to(ROOT)}")
print(f"X shape: {X.shape}")
print(f"number of feature names: {len(feature_names)}")

Loaded: data/correctness_dataset/features/dd_xgb_wl_delta/seed_13/train.npz
X shape: (1156, 55)
number of feature names: 55


In [12]:
for key in feature_file.keys():
    print(key)

X
y
correctness_scores
candidate_ids
domains
splits
problems
corruption_types
feature_names


In [13]:
feature_file["feature_names"]

array(['residual_l2_mean', 'residual_l2_std', 'residual_l2_min',
       'residual_l2_max', 'residual_l2_final', 'residual_l1_mean_mean',
       'residual_l1_mean_std', 'residual_l1_mean_min',
       'residual_l1_mean_max', 'residual_l1_mean_final',
       'cosine_distance_mean', 'cosine_distance_std',
       'cosine_distance_min', 'cosine_distance_max',
       'cosine_distance_final', 'pred_norm_mean', 'pred_norm_std',
       'pred_norm_min', 'pred_norm_max', 'pred_norm_final',
       'pred_delta_norm_mean', 'pred_delta_norm_std',
       'pred_delta_norm_min', 'pred_delta_norm_max',
       'pred_delta_norm_final', 'candidate_delta_norm_mean',
       'candidate_delta_norm_std', 'candidate_delta_norm_min',
       'candidate_delta_norm_max', 'candidate_delta_norm_final',
       'current_goal_l2_mean', 'current_goal_l2_std',
       'current_goal_l2_min', 'current_goal_l2_max',
       'current_goal_l2_final', 'pred_goal_l2_mean', 'pred_goal_l2_std',
       'pred_goal_l2_min', 'pred_goal_l2_

In [14]:
metadata = pd.DataFrame({
    "candidate_id": feature_file["candidate_ids"].astype(str),
    "domain": feature_file["domains"].astype(str),
    "split": feature_file["splits"].astype(str),
    "problem": feature_file["problems"].astype(str),
    "corruption_type": feature_file["corruption_types"].astype(str),
    "correctness_score": correctness_scores,
})
metadata.head()

,candidate_id,domain,split,problem,corruption_type,correctness_score
0,blocks::train::probBLOCKS-4-0::000::gold,blocks,train,probBLOCKS-4-0,gold,1.000000
1,blocks::train::probBLOCKS-4-0::001::truncate,blocks,train,probBLOCKS-4-0,truncate,0.000000
2,blocks::train::probBLOCKS-4-0::002::delete,blocks,train,probBLOCKS-4-0,delete,0.366667
3,blocks::train::probBLOCKS-4-0::003::swap,blocks,train,probBLOCKS-4-0,swap,0.666667
4,blocks::train::probBLOCKS-4-0::004::replace,blocks,train,probBLOCKS-4-0,replace,0.666667


In [17]:
features_df["residual_l2_mean"].describe()

count    1156.000000
mean        1.687707
std         0.482773
min         0.000000
25%         1.295151
50%         1.654743
75%         2.014333
max         3.926548
Name: residual_l2_mean, dtype: float64

In [18]:
features_df["residual_l2_max"].describe()

count    1156.000000
mean        2.105956
std         0.708983
min         0.000000
25%         1.295177
50%         2.373924
75%         2.373927
max         5.823576
Name: residual_l2_max, dtype: float64

In [19]:
features_df["final_goal_l2"].describe()

count    1156.000000
mean        8.548447
std         6.754843
min         0.000000
25%         2.449490
50%         6.164414
75%        12.083046
max        30.033316
Name: final_goal_l2, dtype: float64

In [20]:
features_df["initial_goal_l2"].describe()

count    1156.000000
mean       18.417618
std         7.256278
min         0.000000
25%        12.083046
50%        18.055470
75%        22.045408
max        30.033316
Name: initial_goal_l2, dtype: float64

In [26]:
# The feature matrices are available for every source family, seed, and split.
available = []
for family_dir in sorted(FEATURES.iterdir()):
    if not family_dir.is_dir():
        continue
    for seed_dir in sorted(family_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        available.append({
            "family": family_dir.name,
            "seed": seed_dir.name.replace("seed_", ""),
            "splits": ", ".join(sorted(p.stem for p in seed_dir.glob("*.npz"))),
        })
pd.DataFrame(available).head(12)

,family,seed,splits
0,ad_lstm_wl_delta,13,"test-extrapolation, test-interpolation, train,..."
1,ad_lstm_wl_delta,23,"test-extrapolation, test-interpolation, train,..."
2,ad_lstm_wl_delta,37,"test-extrapolation, test-interpolation, train,..."
3,ad_xgb_wl_delta,13,"test-extrapolation, test-interpolation, train,..."
4,ad_xgb_wl_delta,23,"test-extrapolation, test-interpolation, train,..."
5,ad_xgb_wl_delta,37,"test-extrapolation, test-interpolation, train,..."
6,dd_lstm_shortest_path_delta,13,"test-extrapolation, test-interpolation, train,..."
7,dd_lstm_shortest_path_delta,23,"test-extrapolation, test-interpolation, train,..."
8,dd_lstm_shortest_path_delta,37,"test-extrapolation, test-interpolation, train,..."
9,dd_xgb_wl_delta,13,"test-extrapolation, test-interpolation, train,..."
